# 📘 S8_P2 — KMeans vs DBSCAN sur les « moons »

## 🎯 Montrer où KMeans échoue et DBSCAN réussit
`make_moons` génère deux **croissants entrelacés**. KMeans (clusters ronds) va les couper de travers ; DBSCAN (densité) devrait épouser leur forme. C'est LA démo qui oppose les deux algorithmes.

⚠️ **Note :** ce notebook est **incomplet** (la visualisation DBSCAN manque et `eps=0.1` est trop petit). La version **corrigée et complète** est **S8_P2_v2** (et son cours : `Complet/S8_P2_v2_cours_complet.ipynb`).

---
Imports + génération de 500 points en 2 croissants.

In [30]:
import math
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, DBSCAN
import pandas as pd
import plotly.express as px

In [31]:
X_moons, _ = make_moons(500, noise=0.1, random_state=32)

In [32]:
px.scatter(pd.DataFrame({"x": X_moons[:, 0], "y": X_moons[:, 1]}), x="x", y="y", title="Moon Dataset")

## 🧠 THÉORIE — KMeans : le clustering par centres
On donne le nombre de clusters `k`, et l'algorithme : 1) place k **centres** au hasard, 2) assigne chaque point au centre le plus proche, 3) déplace chaque centre au milieu de ses points, 4) répète jusqu'à stabilité. `fit(X)` puis `.labels_` = le numéro de cluster de chaque point.

**⚠️ Limite clé :** KMeans ne fait que des clusters **ronds/sphériques** — il échoue sur des formes allongées ou entrelacées comme les « moons ».

On demande `n_clusters=2` (on sait qu'il y a 2 croissants).

In [33]:
kmeans = KMeans(n_clusters=2, random_state=32).fit(X_moons)

In [34]:
moons = pd.DataFrame(X_moons)

In [35]:
moons

,0,1
0,0.925283,0.368706
1,0.425729,-0.254396
2,-0.644644,0.659125
3,0.600931,-0.341874
4,-0.706141,0.787363
...,...,...
495,1.136522,-0.500193
496,0.302768,-0.229288
497,1.326780,-0.426990
498,0.751822,0.535959


🔍 Les labels KMeans (0 ou 1) de chaque point.

In [36]:
kmeans.labels_

array([0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1,
       0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0,
       1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1,

## 🔍 Résultat KMeans : l'échec attendu
KMeans coupe les croissants par une frontière quasi droite — il ne fait que des blocs ronds autour de ses 2 centres, il ne « voit » pas les croissants.

In [37]:
results_df = pd.DataFrame({"x": X_moons[:, 0], "y": X_moons[:, 1], "cluster": kmeans.labels_})
px.scatter(results_df, x="x", y="y", color="cluster", title="K-Means Clustering Results")

Les coordonnées des 2 centres trouvés par KMeans.

In [38]:
kmeans.cluster_centers_

array([[ 1.20504508, -0.07892806],
       [-0.20273079,  0.58241751]])

## 🧠 THÉORIE — DBSCAN : le clustering par DENSITÉ
Approche opposée à KMeans : **pas besoin de choisir k** (il le trouve seul), gère les formes **quelconques**, et marque les points isolés comme **bruit** (label **−1**). Deux paramètres : **`eps`** (rayon de voisinage) et **`min_samples`** (voisins minimum pour un cœur de cluster).

**⚠️ DBSCAN est très sensible à `eps`** : trop petit → tout devient du bruit ou plein de mini-clusters.

## 🔴 Deux problèmes dans cette cellule (corrigés dans le v2)
1. **`eps=0.1` est trop petit** : sur ces moons, ça fragmente en ~4 clusters + 32 points de bruit au lieu de 2 croissants propres. Le bon réglage (trouvé par test) est **`eps=0.16, min_samples=5`** → 2 clusters nets.
2. **Il manque la suite** : ce notebook calcule `labels_DB` mais ne construit jamais `labels_df` ni ne **trace** le résultat DBSCAN — donc on ne voit pas que DBSCAN réussit là où KMeans échoue. (L'erreur `NameError` affichée est une trace périmée d'une version antérieure.)

👉 Ouvre **S8_P2_v2** pour la version qui tourne de bout en bout et affiche les deux graphiques.

## 📝 Résumé
KMeans = clusters ronds (échoue sur les moons) · DBSCAN = par densité, formes quelconques, détecte le bruit (réussit, une fois `eps` bien réglé). La leçon centrale du clustering.

In [ ]:
dbscan = DBSCAN(eps=0.1, min_samples=5).fit(X_moons)
labels_DB = dbscan.labels_

NameError: name 'labels_DB' is not defined